In [0]:
import pyspark.sql.functions as F

def create_customer_product(order_table,cust_table,prod_table):
    # 1. Load your clean silver tables
    orders_df = spark.table(order_table)
    customers_df = spark.table(cust_table)
    products_df = spark.table(prod_table)

    customers_df=customers_df.drop("state")

    # 2. CREATE A LEAN BRIDGE: Extract ONLY the unique relationship mappings
    # This strips out dates, quantities, and prices, leaving a highly optimized mapping layer
    customer_product_bridge = orders_df.select("CustomerID", "ProductID").distinct()

    # 3. Join the descriptive attributes onto your lean bridge
    enriched_cust_prod_df = customer_product_bridge \
        .join(customers_df, on="CustomerID", how="inner") \
        .join(products_df, on="ProductID", how="inner")

    return enriched_cust_prod_df

In [0]:
def create_enriched_orders(order_table,cust_table,prod_table):

    orders_df = spark.table(order_table)
    customers_df = spark.table(cust_table)
    products_df = spark.table(prod_table)

    customers_df=customers_df.drop("state")

    #Enrich the orders data by joining customer and product details
    enriched_df = orders_df \
        .join(customers_df, on="CustomerID", how="left") \
        .join(products_df, on="ProductID", how="left")

    return enriched_df

In [0]:
import pyspark.sql.functions as F

def create_enriched_sales(order_table,cust_table,prod_table,target_table):
    """
    Enriches order data with Customer and Product context, 
    rounds profits, and writes to the Gold layer for BI reporting.
    """
    # 1. Load clean, validated data from your Silver layer
    orders_df = spark.table(order_table)
    customers_df = spark.table(cust_table)
    products_df = spark.table(prod_table)

    # 2. Join customer and product dimensional details onto the core order stream
    enriched_df = orders_df \
        .join(customers_df, on="CustomerID", how="left") \
        .join(products_df, on="ProductID", how="left")

    # 3. Apply Transformations: Round Profit and organize columns cleanly
    final_reporting_df = enriched_df \
        .withColumn("Profit", F.round(F.col("Profit"), 2)) \
        .select(
            # --- Core Order Information ---
            "OrderID",
            "OrderDate",
            "RowID",
            "Quantity",
            "Price",
            "Discount",
            "ShipDate",
            "ShipMode",
            
            # --- Transformed Financial Metric ---
            "Profit",
            
            # --- Enriched Customer Context ---
            "CustomerName",
            "Country",
            
            # --- Enriched Product Context ---
            "Category",
            "SubCategory",
            
            # --- Tracking Keys (Kept for downstream relationships/lineage) ---
            "CustomerID",
            "ProductID"
        )

    return final_reporting_df

In [0]:
import pyspark.sql.functions as F

def create_gold_profit_aggregation(input_table):
    """
    Aggregates total profit by Year, Category, Sub-Category, and Customer.
    Explicitly handles the 'd/M/yyyy' string date format to prevent casting errors.
    """
    # 1. Load the pre-joined enriched data from your Gold layer
    enriched_df = spark.table(input_table)

    # 2. Fix the date format on-the-fly and extract the Year safely
    # F.to_date converts '21/8/2016' into a true Spark Date object, allowing F.year to work perfectly.
    aggregated_df = enriched_df \
        .withColumn("Year", F.year(F.to_date(F.col("OrderDate"), "d/M/yyyy"))) \
        .groupBy(
            "Year",
            "Category",
            "SubCategory",
            "CustomerID",   
            "CustomerName"
        ) \
        .agg(
            F.round(F.sum(F.col("Profit")), 2).alias("TotalProfit")
        ) \
        .select(
            "Year",
            "Category",
            "SubCategory",
            "CustomerID",
            "CustomerName",
            "TotalProfit"
        ) \
        .orderBy("Year", "Category", "SubCategory", F.col("TotalProfit").desc())

    return aggregated_df

Profit by Year

In [0]:
def aggregate_profit_by_year(source_table):
    """
    Computes total profit aggregated by calendar year.
    Saves to: assessment.gold.report_profit_by_year
    """
    print("Computing Matrix: Profit by Year...")
    
    sql_query = f"""
        SELECT 
            YEAR(to_date(OrderDate, 'd/M/yyyy')) AS OrderYear,
            ROUND(SUM(Profit), 2) AS TotalProfit
        FROM {source_table}
        GROUP BY YEAR(to_date(OrderDate, 'd/M/yyyy'))
        ORDER BY OrderYear DESC
    """
    
    df = spark.sql(sql_query)

    return df

Profit by Year + Product Category

In [0]:
def aggregate_profit_by_year_category(source_table):
    """
    Computes total profit aggregated by year and product category.
    Saves to: assessment.gold.report_profit_by_year_category
    """
    print("Computing Matrix: Profit by Year + Product Category...")
    
    sql_query = f"""
        SELECT 
            YEAR(to_date(OrderDate, 'd/M/yyyy')) AS OrderYear,
            Category,
            ROUND(SUM(Profit), 2) AS TotalProfit
        FROM {source_table}
        GROUP BY 
            YEAR(to_date(OrderDate, 'd/M/yyyy')), 
            Category
        ORDER BY 
            OrderYear DESC, 
            TotalProfit DESC
    """
    
    df = spark.sql(sql_query)

    return df

Profit by Customer

In [0]:
def aggregate_profit_by_customer(source_table):
    """
    Computes total lifetime profit aggregated by individual customer.
    Saves to: assessment.gold.report_profit_by_customer
    """
    print("Computing Matrix: Profit by Customer...")
    
    sql_query = f"""
        SELECT 
            CustomerID,
            CustomerName,
            ROUND(SUM(Profit), 2) AS TotalProfit
        FROM {source_table}
        GROUP BY 
            CustomerID, 
            CustomerName
        ORDER BY 
            TotalProfit DESC
    """
    
    df = spark.sql(sql_query)

    return df

Profit by Customer + Year

In [0]:
def aggregate_profit_by_customer_year(source_table):
    """
    Computes breakdown of total profit aggregated by customer per calendar year.
    Saves to: assessment.gold.report_profit_by_customer_year
    """
    print("Computing Matrix: Profit by Customer + Year...")
    
    sql_query = f"""
        SELECT 
            CustomerID,
            CustomerName,
            YEAR(to_date(OrderDate, 'd/M/yyyy')) AS OrderYear,
            ROUND(SUM(Profit), 2) AS TotalProfit
        FROM {source_table}
        GROUP BY 
            CustomerID, 
            CustomerName, 
            YEAR(to_date(OrderDate, 'd/M/yyyy'))
        ORDER BY 
            CustomerName ASC, 
            OrderYear DESC
    """
    
    df = spark.sql(sql_query)
    return df

In [0]:
def publish_write(order_table,cust_table,prod_table,order_summary,target_table):
    if target_table=="assessment.gold.enriched_customer_products":
        enriched_df=create_customer_product(order_table,cust_table,prod_table)
    elif target_table=="assessment.gold.enriched_sales_analysis":
        enriched_df=create_enriched_orders(order_table,cust_table,prod_table)
    elif target_table=="assessment.gold.enriched_orders_summary":
        enriched_df=create_enriched_sales(order_table,cust_table,prod_table,target_table)
    elif target_table=="assessment.gold.aggregated_profit_summary":
        enriched_df=create_gold_profit_aggregation(order_summary)
    elif target_table=="assessment.gold.report_profit_by_year":
        enriched_df=aggregate_profit_by_year(order_summary)
    elif target_table=="assessment.gold.report_profit_by_year_category":
        enriched_df=aggregate_profit_by_year_category(order_summary)
    elif target_table=="assessment.gold.report_profit_by_customer":
        enriched_df=aggregate_profit_by_customer(order_summary)
    elif target_table=="assessment.gold.report_profit_by_customer_year":
        enriched_df=aggregate_profit_by_customer_year(order_summary)
    
    #Save the final relationship table to Gold
    print(f"Saving enriched Gold Layer relationship table for target {target_table}...")
    enriched_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    print("Enrichment complete.")

In [0]:
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.enriched_customer_products")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.enriched_sales_analysis")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.enriched_orders_summary")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.aggregated_profit_summary")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.report_profit_by_year")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.report_profit_by_year_category")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.report_profit_by_customer")
# publish_write("assessment.silver.orders_dqc_accepted","assessment.silver.customer_dqc_accepted","assessment.silver.product_dqc_accepted","assessment.gold.enriched_orders_summary","assessment.gold.report_profit_by_customer_year")
